# E2′ — domain diagnostic on the S6-out TRAIN features (C1_s6out)

The fourth replication of the §9 train-feature domain diagnostic
(pre-registered, §10.3), and the cleanest test of the structural root
cause found on p2_lab: this is the first train set whose second
environment is a whole non-bedroom domain other than the living room —
**AR-7 (laboratory, 11 traces) is IN train** here, S6 is the held-out
test.

Instrument: `sharp_har.diagnostics.domain_probe` — promoted to the
package 2026-07-18, math verified **byte-identical** to the
notebook-local cell the three executed C1/C2/C3 sessions ran (see
STATUS). Importing it, instead of pasting a fourth copy, is what keeps
this run comparable to the recorded rows and cross-reviewable
(CLAUDE.md: no logic in notebooks).

One GPU session, two steps:

1. **cache the TRAIN features** of `C1_s6out/best.ckpt` over the frozen
   `splits/p2_living.json` — no cache exists yet for this rotation
   (the C1/C2/C3 sessions reused probe-session caches; there is no
   probe session on E2′, so the cache is produced here, train only);
2. **run the domain probe** on that cache.

Train features only — no val selection, no test contact (§0.7).
Declared confound, same as the three recorded sessions: the encoder
trained on these traces, so the `y` control saturates and absolute
numbers are memorization-inflated — the reading is each target's
delta vs its OWN majority baseline.

**Expected readings** (STATUS gets one line either way):

- control `y` ≈ 1 with a large positive delta — else the session is
  void, fix before reading anything else;
- domain targets at/below their majority baselines → the "GRL had no
  target" structural verdict replicates with a laboratory second
  environment;
- any domain target clearly ABOVE its baseline would be the first
  counterexample across four train sets — a report-level finding:
  stop and discuss before any further step.

Executed copy → `notebooks/diagnostics/YYYY-MM-DD_c1_s6out_domain_probe.ipynb`
(next to the other three domain-probe sessions) + STATUS line, same
commit. This template stays output-free.

## Environment setup

Full preamble incl. data staging (step 1 runs the frozen encoder over
the real staged data). GPU runtime recommended (~2 min of forwards;
CPU works but is slow).

In [1]:
from pathlib import Path
import subprocess
import sys
import torch
import zipfile

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
elif (cwd.parent.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

# After the clone, so the file actually exists on a fresh runtime.
!pip install -q -r {REPO_DIR}/requirements.txt

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

drive.mount("/content/drive")  # idempotent

# Stage the zip archives if needed (same convention as 00/03/04).
if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print(f"copying {src} -> {dst}")
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo dir:", REPO_DIR)
print("GPU available:", torch.cuda.is_available())
print("Stage dir:", stage_dir)
print("Checkpoint root:", CKPT_ROOT)

Mounted at /content/drive
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces.zip -> /content/doppler_traces.zip
copying /content/drive/MyDrive/DATASET_SHARP/doppler_traces_S4_S5.zip -> /content/doppler_traces_S4_S5.zip
Repo dir: /content/sharp-har
GPU available: True
Stage dir: /content/data
Checkpoint root: /content/drive/MyDrive/sharp_har_runs


## Step 1 — cache TRAIN features from C1_s6out/best.ckpt

`harness.cache_features` over the frozen `splits/p2_living.json` train
set (80 traces — twin binding included, see the split session).
Re-running is safe: an existing file is overwritten by a fresh,
identical extraction (frozen encoder, no augmentation, §5.3 note i).
Expected: **50948 (window, antenna) samples, d=256** (train stride 100,
counts derived from the frozen inventory + split — blocking asserts
below).

In [2]:
import numpy as np
from sharp_har.harness import cache_features

CKPT = CKPT_ROOT / "C1_s6out" / "best.ckpt"
SPLIT = REPO_DIR / "splits" / "p2_living.json"
assert CKPT.exists(), f"{CKPT} not found - is the C1_s6out Drive folder yours or shortcut-linked?"
assert SPLIT.exists(), f"{SPLIT} not found - pull the repo: the frozen split landed 2026-07-18"

cache_path = cache_features(CKPT, SPLIT, "train", stage_dir=stage_dir, repo_dir=REPO_DIR)
d = np.load(cache_path, allow_pickle=False)
n, dim = d["features"].shape
assert n == 50948, f"{n} samples, expected 50948 (frozen p2_living train at stride 100)"
assert dim == 256, f"d={dim}, expected 256"
assert len(set(d["trace_id"].tolist())) == 80, "expected the 80 frozen train traces"
print(f"train cache: {n} samples, d={dim} -> {cache_path}")

2026-07-18 18:57:26,238 [INFO] sharp_har.data: train: 80 traces, 50948 (window, antenna) samples (win=340, stride=100)
2026-07-18 18:58:07,237 [INFO] sharp_har.harness: cached 50948 features (d=256) -> /content/drive/MyDrive/sharp_har_runs/C1_s6out/features_best_train.npz


train cache: 50948 samples, d=256 -> /content/drive/MyDrive/sharp_har_runs/C1_s6out/features_best_train.npz


## Step 2 — run the domain probe

`diagnostics.domain_probe`: frozen §5.3 recipe on an inner
trace-disjoint stratified split of the 80 train traces, every target
against its own majority baseline, positive control `y` first. Note
the AR-set space of this rotation is {AR-1…AR-5, AR-7} and
`ambiente` is bedroom vs **laboratory** (on p2_lab it was bedroom vs
living room) — the target spaces come from the cache itself, nothing
to configure.

In [3]:
import pandas as pd
from sharp_har.diagnostics import domain_probe

rows = domain_probe(cache_path, "C1_s6out")

df = pd.DataFrame(rows).drop(columns=["labels"])
display(df)

print("READ THE CONTROL FIRST: y must sit near 1.0 with a large positive")
print("delta (the three recorded sessions: 1.000 / 0.893 / ~0.995) - else the")
print("inner split is broken and no other row means anything.")
print()
print("Only if the control passes: delta ~ 0 on every domain target replicates")
print("the structural verdict with a laboratory second environment; a clearly")
print("positive delta on any domain target is the FIRST counterexample - stop")
print("and bring it to the team before anything else.");

2026-07-18 18:58:46,012 [INFO] sharp_har.diagnostics: [C1_s6out] domain probe: 50948 samples (d=256), inner split 53 fit / 27 eval traces (33116 / 17832 samples)
2026-07-18 18:58:58,587 [INFO] sharp_har.probe: linear probe [y]: best val macro-F1 0.8523 at epoch 17/22
2026-07-18 18:58:58,762 [INFO] sharp_har.diagnostics: [C1_s6out] y            (8 cls): acc 0.870 vs baseline 0.210 (delta +0.660), macro-F1 0.852  <- CONTROL
2026-07-18 18:59:01,488 [INFO] sharp_har.probe: linear probe [ar_set]: best val macro-F1 0.1598 at epoch 2/7
2026-07-18 18:59:01,713 [INFO] sharp_har.diagnostics: [C1_s6out] ar_set       (6 cls): acc 0.321 vs baseline 0.310 (delta +0.011), macro-F1 0.160
2026-07-18 18:59:03,797 [INFO] sharp_har.probe: linear probe [ambiente]: best val macro-F1 0.4723 at epoch 1/6
2026-07-18 18:59:03,950 [INFO] sharp_har.diagnostics: [C1_s6out] ambiente     (2 cls): acc 0.895 vs baseline 0.895 (delta +0.000), macro-F1 0.472
2026-07-18 18:59:08,349 [INFO] sharp_har.probe: linear probe [

,run,target,n_classes,eval_accuracy,majority_baseline,delta,macro_f1,best_epoch,classes_present_in_eval
0,C1_s6out,y,8,0.869672,0.209735,0.659937,0.852299,17,8
1,C1_s6out,ar_set,6,0.320772,0.309556,0.011216,0.159774,2,6
2,C1_s6out,ambiente,2,0.895020,0.895020,0.000000,0.472301,1,2
3,C1_s6out,direct_path,2,0.683042,0.712203,-0.029161,0.498345,11,2
4,C1_s6out,persona,3,0.670480,0.670480,0.000000,0.267580,1,3
5,C1_s6out,monitor,3,0.608793,0.607223,0.001570,0.360123,11,3


READ THE CONTROL FIRST: y must sit near 1.0 with a large positive
delta (the three recorded sessions: 1.000 / 0.893 / ~0.995) - else the
inner split is broken and no other row means anything.

Only if the control passes: delta ~ 0 on every domain target replicates
the structural verdict with a laboratory second environment; a clearly
positive delta on any domain target is the FIRST counterexample - stop
and bring it to the team before anything else.


## Archiving

Download the executed copy and commit it verbatim to
`notebooks/diagnostics/` as `YYYY-MM-DD_c1_s6out_domain_probe.ipynb`
(+ README index row + STATUS line, same commit). This template stays
output-free.